# linkedin-content-agent content pipeline on Colab

Sets up Ollama + this app's own backend only -- **no ComfyUI, no character-forge-v2 cloned here at all**. Image generation is submitted to the shared [`task-queue`](https://github.com/digishgabhawala/task-queue) service and a *separate* worker (anywhere -- this machine, a friend's laptop, or `character-forge-v2`'s own Colab notebook running in a completely different session) claims and renders it. This notebook only ever talks to the queue's public API.

**Before running the image-generation section below**: make sure some image worker is actually running against the same queue (e.g. open `character-forge-v2/notebooks/colab_character_forge.ipynb` in a second Colab session and run it through its own worker section). Without a worker polling, a submitted task just sits `queued` forever -- there's no error for this, it's a valid state.

**Before running:** Runtime -> Change runtime type -> GPU is **not** required for this notebook (Ollama can run CPU-only, just slower) -- but a GPU-enabled runtime works fine too if that's what you have available.

Repos:
- https://github.com/digishgabhawala/linkedin-content-agent
- https://github.com/digishgabhawala/task-queue (used only as a remote API here -- not cloned)

## 1. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/digishgabhawala/linkedin-content-agent.git

## 2. Ollama + qwen3:14b

linkedin-content-agent's clarify/draft/judge/scene calls all go through Ollama. The `qwen3:14b` pull is a substantial download (~9GB).

### Before pulling qwen3:14b: free up disk space

Found live during earlier testing: Colab's disk fills up faster than expected once pip/apt caches are counted. Safe cleanup first (reclaims caches only, nothing here touches anything still needed):

In [ ]:
!apt-get clean
!pip cache purge
!rm -rf /content/sample_data
!df -h /content

Also found live: Ollama's install script needs `zstd` to decompress itself, which Colab's base image doesn't ship:

In [ ]:
!apt-get -qq install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess

with open("/content/ollama.log", "w") as ollama_log:
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT,
        start_new_session=True,  # survives interrupting other notebook cells
    )
print("Ollama starting (pid", ollama_proc.pid, ")")

In [ ]:
import time

import requests

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:11434", timeout=3).status_code == 200:
            print("Ollama is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama did not come up -- check /content/ollama.log")

In [ ]:
!ollama pull qwen3:14b

## 3. linkedin-content-agent backend

Installed into its own venv -- `python3 -m venv` reliably fails on Colab's Python image (`ensurepip` bootstrap issue found live), so this uses `virtualenv` instead, which bundles its own pip.

In [ ]:
%cd /content/linkedin-content-agent/backend
!pip install -q virtualenv
!virtualenv -q .venv
!.venv/bin/pip install -q -r requirements.txt

`TASK_QUEUE_URL` points at the real deployed queue by default -- change it if you're pointing at a different instance.

In [ ]:
with open(".env", "w") as f:
    f.write("TASK_QUEUE_URL=https://task-queue-eight.vercel.app\n")

with open(".env") as f:
    print(f.read())

In [ ]:
import subprocess

with open("/content/backend.log", "w") as backend_log:
    backend_proc = subprocess.Popen(
        [".venv/bin/python", "-m", "uvicorn", "app.main:app", "--port", "11000"],
        stdout=backend_log, stderr=subprocess.STDOUT,
        start_new_session=True,  # survives interrupting other notebook cells
    )
print("Backend starting (pid", backend_proc.pid, ") -- log at /content/backend.log")

In [ ]:
import time

import requests

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11000/api/health", timeout=3).status_code == 200:
            print("Backend is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Backend did not come up -- check /content/backend.log")

## 4. Drive the real pipeline through the JSON API

No frontend/tunnel needed to validate the actual brief -> post -> image loop -- everything the React UI does is just calls to this same API. (The frontend itself was already validated locally against this same API contract.)

Edit `BRIEF` below to whatever you want to test with -- a richer brief (specific numbers, what was tried, why it mattered) is more likely to sail through scoring without a clarify/escalation round-trip; a thin one is a good test of the clarify + thin-material-gate behavior described in the main README.

In [ ]:
import requests

API = "http://127.0.0.1:11000/api"

BRIEF = (
    "Just decoupled image generation from this app into a separate task-queue "
    "service -- ComfyUI and Ollama kept fighting over RAM when run in the same "
    "process, so now image generation happens on a completely separate worker "
    "machine, coordinated through a public queue on Vercel + Supabase."
)

post = requests.post(f"{API}/posts", json={"brief": BRIEF}).json()
print("post id:", post["id"], "| status:", post["status"])
post_id = post["id"]

In [ ]:
# Clarify loop: answer questions until it leaves "clarifying". Run this cell
# repeatedly (or wrap it in a while loop) if it asks more than one question.
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])

if post["status"] == "clarifying" and post["pending_question"]:
    print("\nQuestion:", post["pending_question"])
    answer = input("Your answer: ")
    post = requests.post(f"{API}/posts/{post_id}/clarify", json={"answer": answer}).json()
    print("\nnew status:", post["status"])
else:
    print("No pending question -- re-run this cell later if status is still 'clarifying',\n"
          "otherwise move on to the next cell.")

**If status is `needs_input`**: a scoring gate that a rewrite can't fix (not enough material, or the wrong angle) escalated back to you -- see `post["escalation_reason"]`. Either provide more detail (`POST /posts/{id}/additional-info {"info": "..."}`) or accept the current best draft as-is (`POST /posts/{id}/accept-draft`, no body).

In [ ]:
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])

if post["status"] == "needs_input":
    print("\nEscalation reason:", post["escalation_reason"])
    choice = input("Type more detail to add, or leave blank to accept the current draft as-is: ")
    if choice.strip():
        post = requests.post(f"{API}/posts/{post_id}/additional-info", json={"info": choice}).json()
    else:
        post = requests.post(f"{API}/posts/{post_id}/accept-draft", json={}).json()
    print("\nnew status:", post["status"])
else:
    print("Not in needs_input -- nothing to do here.")

In [ ]:
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])
print("category:", post["category"], "| weighted_score:", post["weighted_score"])
print("\n--- draft ---\n")
print(post["post_text"])
print("\n--- pillar scores ---")
for pillar, s in {**post["gate_scores"], **post["pillar_scores"]}.items():
    print(f"  {pillar}: {s['score']}/10 -- {s['reason']}")

## 5. Lock and generate the image

Locking derives a scene description (and reuses/creates an `@name` recurring backdrop asset if the scene proposes one). Generating submits a task to the queue and returns immediately -- the real 15-40 min render happens wherever a worker picks it up, not in this process. This cell polls until it's done.

In [ ]:
post = requests.post(f"{API}/posts/{post_id}/lock", json={}).json()
print("status:", post["status"])
print("scene_instruction:", post["scene_instruction"])
print("scene_asset_name:", post["scene_asset_name"])

In [ ]:
post = requests.post(f"{API}/posts/{post_id}/generate-image", json={}).json()
print("status:", post["status"])

In [ ]:
import time

start = time.time()
while True:
    post = requests.get(f"{API}/posts/{post_id}").json()
    elapsed = int(time.time() - start)
    print(f"[{elapsed}s] status: {post['status']}")
    if post["status"] in ("image_ready", "image_failed"):
        break
    if post["is_stalled"]:
        print("  (flagged as possibly stalled -- check that a worker is actually running "
              "against the same queue)")
    time.sleep(30)

if post["status"] == "image_failed":
    print("\nFAILED:", post["image_job_error"])
else:
    print("\nDone.")

In [ ]:
from IPython.display import Image, display

if post["status"] == "image_ready":
    print(post["post_text"])
    # image_url is a full external URL (Supabase Storage, via the task-queue's
    # artifact store) for a real render, or a path served by this app's own
    # /generated mount for the placeholder image -- handle both.
    url = post["image_url"]
    if not url.startswith("http"):
        url = f"http://127.0.0.1:11000{url}"
    display(Image(url=url))

## What this proves (and doesn't)

**Proves:** the actual decoupled architecture working end to end -- this notebook never clones character-forge-v2 or touches ComfyUI at all, yet a real image gets rendered and comes back, because *some other worker* (potentially a completely separate Colab session, potentially a different physical machine entirely) claimed the task from the shared queue and did the work. That's the point: this notebook and the image worker have zero knowledge of each other beyond the queue's public URL.

**Doesn't cover:** the React frontend itself (this notebook drives the API directly -- the UI was already validated locally against this same API contract), the `lightning` speed profile, or feedback consumption (not built yet, see the README's Upcoming section).